Jaincelle Choi Ham
400249037
Link to Github Repository:https://github.com/jaincelleham/O712Assignment
Link to Streamlit App: https://o712assignment-hazohvc2maueb4bui8gi3w.streamlit.app/


# STARTER CODE AND IMPORTS


In [16]:
%pip install -q git+https://github.com/zhouy185/rosa-starter.git

Note: you may need to restart the kernel to use updated packages.


In [17]:
import numpy as np
from starter import ZONES, TIME_BLOCKS, COSTS, PROMISE, delivery_times

In [18]:
times = delivery_times("Far West", "Fri/Sat eve", 45, seed = 1)
print(len(times), "orders; average delivery time", round(times.mean(), 1),
"minutes")

210 orders; average delivery time 42.9 minutes


In [19]:
print(ZONES)
print(TIME_BLOCKS)
print(COSTS)
print(PROMISE)

['Central', 'North', 'Far West']
['Lunch', 'Weekday eve', 'Fri/Sat eve', 'Other']
{'refund': 10.0, 'churn_orders': 1.8, 'margin': 9.0}
45


# Part I

## a. Percentage of late orders

In [20]:
def percentage_late(zone, time_block, promise):
  if zone == "all":
    zones = ZONES
  else:
    zones = [zone]
  if time_block == "all":
    time_blocks = TIME_BLOCKS
  else:
    time_blocks = [time_block]

  # Count how many late orders there are and how many orders there were in total
  late_count = 0
  total_count = 0
  for z in zones:
    for t in time_blocks:
      times = delivery_times(z, t, promise, seed = 1)
      for d in times: # d is short for each delivery's time
        if d > promise:
          late_count = late_count + 1
      total_count = total_count + len(times)

  return round(float(late_count / total_count * 100),2)

In [21]:
# Test: one example for each case
print (f"Far West, Fri/Sat eve: {percentage_late('Far West', 'Fri/Sat eve', 45):.2f}%")
print (f"all, Lunch: {percentage_late('all', 'Lunch', 45):.2f}%")
print (f"Far West, all: {percentage_late('Far West', 'all', 45):.2f}%")
print (f"all, all: {percentage_late('all', 'all', 45):.2f}%")

Far West, Fri/Sat eve: 36.67%
all, Lunch: 3.33%
Far West, all: 15.61%
all, all: 8.29%


## b. Ranking by late arrival rate (descending order)

In [22]:
results = []
for z in ZONES:
  for t in TIME_BLOCKS:
    pct = percentage_late(z, t, 45)
    results.append((pct, z, t))

# Sort the results by late rate, descending order
results.sort(reverse=True) # Note: Gemini suggested this line and after evaluating, I accepted
for pct, z, t in results:
  print(f"{z},{t}: {pct:.2f}%")

Far West,Fri/Sat eve: 36.67%
North,Fri/Sat eve: 16.67%
Far West,Weekday eve: 11.18%
Central,Fri/Sat eve: 6.84%
North,Weekday eve: 6.00%
Far West,Other: 4.69%
Central,Weekday eve: 4.24%
North,Lunch: 3.68%
Central,Lunch: 3.46%
Central,Other: 2.63%
North,Other: 2.50%
Far West,Lunch: 2.50%


## c. Average delivery time

In [23]:
def average_delivery_time(zone, time_block, promise):
  if zone == "all":
    zones = ZONES
  else:
    zones = [zone]
  if time_block == "all":
    time_blocks = TIME_BLOCKS
  else:
    time_blocks = [time_block]

# Running totals for minutes and number of deliveries
  total_minutes = 0
  total_count = 0
  for z in zones:
    for t in time_blocks:
      times = delivery_times(z, t, promise, seed = 1)
      total_minutes += times.sum() # Gemini suggested the next 2 lines
      total_count += len(times)

  return round(float(total_minutes / total_count),2)

In [24]:
# Test: an example for each case
print (f"Far West, Fri/Sat eve: {average_delivery_time('Far West', 'Fri/Sat eve', 45):.2f} minutes")
print (f"all, Lunch: {average_delivery_time('all', 'Lunch', 45):.2f} minutes")
print (f"Far West, all: {average_delivery_time('Far West', 'all', 45):.2f} minutes")
print (f"all, all: {average_delivery_time('all', 'all', 45):.2f} minutes")

Far West, Fri/Sat eve: 42.92 minutes
all, Lunch: 28.22 minutes
Far West, all: 35.27 minutes
all, all: 31.19 minutes


## d. Ranking by average delivery time (descending order)

In [25]:
results = []
for z in ZONES:
  for t in TIME_BLOCKS:
    avg = average_delivery_time(z, t, 45)
    results.append((avg, z, t))

# Sort by average delivery time, descending order
results.sort(reverse=True)
for avg, z, t in results:
  print(f"{z},{t}: {avg:.2f} minutes")

Far West,Fri/Sat eve: 42.92 minutes
North,Fri/Sat eve: 36.77 minutes
Far West,Weekday eve: 35.04 minutes
Central,Fri/Sat eve: 32.13 minutes
North,Weekday eve: 31.38 minutes
Far West,Lunch: 30.58 minutes
Far West,Other: 30.04 minutes
Central,Weekday eve: 29.03 minutes
North,Lunch: 28.67 minutes
North,Other: 27.66 minutes
Central,Lunch: 26.80 minutes
Central,Other: 26.04 minutes


## e. Late rate vs average delivery time, which is more relevant to Rosa's decisions

In [26]:
# For Rosa's decisions, the late rate is more relvant.
# With every late order, Rosa would not only have to pay a $10 refund but also lose future orders.
# Basically, an order at 44 minutes costs her nothing, but even one minute over the agreed time of 45 minutes costs her money.
# Also, the average can hide the problem area.
# For instance, Far West on Friday and Saturday evenings averages at 42.92 minutes, which is under the agreed 45 minutes.
# However, 36.67% of those orders are late, which is the worst of any zone and time block.

# PART II

## a. Cost per late order

In [27]:
def cost_per_late_order(costs):
  refund = costs["refund"] # cost 1: money Rosa pays back per late order
  lost_profit = costs["churn_orders"] * costs["margin"] # cost 2: future profit lost
  return refund + lost_profit # total cost of one late order

print(f"Cost per late order: ${cost_per_late_order(COSTS):.2f}")


Cost per late order: $26.20


## b. Choosing the best promise

In [28]:
def net_profit(zone, time_block, promise, costs):
  times = delivery_times(zone, time_block, promise, seed = 1)

# Profit collected from all orders
  orders = len(times)
  profit = orders * costs["margin"]

# Cost of late orders (refund + lost future profit)
  late_count = (times > promise).sum()
  late_cost = late_count * cost_per_late_order(costs)

  return profit - late_cost

# Example with specific values
print(f"Net profit for Far West, Fri/Sat eve, 45 mins: ${net_profit('Far West', 'Fri/Sat eve', 45, COSTS):.2f}")

Net profit for Far West, Fri/Sat eve, 45 mins: $-127.40


In [29]:
# Try each promise and return the one with the highest net profit
def best_promise(zone, time_block, promise, costs, seed = 1):
  best_promise = promise[0]
  best_profit = net_profit(zone, time_block, best_promise, costs)

  for p in promise:
    profit = net_profit(zone, time_block, p, costs)
    if profit > best_profit:
      best_promise = p
      best_profit = profit
  return best_promise, round(float(best_profit),2)

# Promises to try: 5, 10, 15...., 50 minutes
possible_promises = list(range(5, 55, 5))

# Find the best promise for every zone and time block pair
for z in ZONES:
  for t in TIME_BLOCKS:
    p, profit = best_promise(z, t, possible_promises, COSTS)
    print(f"Best promise for {z}, {t}: {p} minutes, net profit: ${profit:.2f}")

Best promise for Central, Lunch: 45 minutes, net profit: $2104.20
Best promise for Central, Weekday eve: 45 minutes, net profit: $2603.20
Best promise for Central, Fri/Sat eve: 50 minutes, net profit: $1399.00
Best promise for Central, Other: 45 minutes, net profit: $1579.00
Best promise for North, Lunch: 45 minutes, net profit: $1526.60
Best promise for North, Weekday eve: 45 minutes, net profit: $1857.00
Best promise for North, Fri/Sat eve: 50 minutes, net profit: $996.40
Best promise for North, Other: 45 minutes, net profit: $1335.20
Best promise for Far West, Lunch: 45 minutes, net profit: $1001.40
Best promise for Far West, Weekday eve: 50 minutes, net profit: $1210.80
Best promise for Far West, Fri/Sat eve: 50 minutes, net profit: $827.40
Best promise for Far West, Other: 45 minutes, net profit: $1492.20


In [30]:
# Test an example
z = "Far West"
t = "Fri/Sat eve"
p, profit = best_promise(z, t, possible_promises, COSTS)
print(f"Best promise for Far West, Fri/Sat eve: {p} minutes, net profit: ${profit:.2f}")

Best promise for Far West, Fri/Sat eve: 50 minutes, net profit: $827.40
